<table>
  <tr>
    <td style="text-align: center">
      <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2Fnotebook%2F06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/notebook/06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
      </a>
    </td>
    <td style="text-align: center">
      <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/notebook/06_BigQuery_Conversational_Analytics_Agent.ipynb">
        <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
      </a>
    </td>
  </tr>
</table>

# Track 1 (Notebook 06 — Lab 1.6): BigQuery Conversational Analytics (BQCA) Data Agent
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

<p align="center">
  <img src="https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook5_bq_graph_and_bqca_flow.png" alt="Track 1 Notebook 06 Architecture: BigQuery Conversational Analytics Data Agent" width="95%">
</p>

---

### 📋 Notebook 06 Step-by-Step Execution Summary
1. **Step 0 (Environment Setup & APIs)**: Auto-detect active `PROJECT_ID` and Singapore region (`asia-southeast1`), and enable `bigquery.googleapis.com`, `geminidataanalytics.googleapis.com`, `cloudaicompanion.googleapis.com`, and `dataplex.googleapis.com`.
2. **Step 1 (Invoke `.sql` Bootstrap Files)**: Ensure `acsm_bronze`, `acsm_silver`, and `acsm_gold` exist (so this notebook can run standalone on standard On-Demand pricing) and execute [`07_bqca_semantic_marts.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/07_bqca_semantic_marts.sql) to materialize the Gold Semantic Marts (`gold_underwriting_funnel`, `gold_collections_risk`) and Graph Node/Edge tables.
3. **Step 2 (Inspect Governed BQCA Semantic Configuration & Gold Marts)**: Audit [`track3_self_serve_analytics/bqca_agent_config.yaml`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track3_self_serve_analytics/bqca_agent_config.yaml) and verify row counts across the governed Gold Marts (`gold_aeon_customer360_profile`, `gold_underwriting_funnel`, `gold_collections_risk`, and `recon_audit_log`).
4. **Step 3 (Create & Publish BigQuery Conversational Analytics Data Agents)**: Programmatically provision and publish governed BQCA Data Agents (`acsm-aeon360-bqca-agent` & `acsm-credit-graph-bqca-agent`) via `geminidataanalytics.googleapis.com/v1beta`, grounded on `acsm_gold` tables, Dataplex Knowledge Graph `schemaRelationships`, ACSM Domain Glossary (`CTOS`, `AKPK`, `DSR`, `Unpaid_OSP`, `FinPlus`), and Verified Golden SQL/GQL queries (`C1.1.2.9`, `C1.1.2.10`, `M1.4.2`, `M1.4.4`).
5. **Step 4 (Live Multi-Turn Conversational Q&A over Gold Marts)**: Execute live natural-language business questions against `acsm-aeon360-bqca-agent` (`:chat` API) returning governed executive summaries, transparent SQL queries, and live BigQuery result frames.
6. **Step 5 (Natural-Language Merchant Contagion & Graph Relationship Q&A)**: Query `acsm-credit-graph-bqca-agent` (`:chat` API) in plain English to identify merchant delinquency contagion and linked unpaid principal (`Unpaid_OSP`).

---
## Step 0: Configure Parameters (`PROJECT_ID` & Singapore Region) & Enable Gemini Data Analytics APIs
This notebook dynamically detects your active Google Cloud project (`PROJECT_ID`), runs in **`asia-southeast1` (Singapore)**, and configures the `global` control plane endpoint for the **Gemini Data Analytics / BigQuery Conversational Analytics API** (`geminidataanalytics.googleapis.com`).

In [ ]:
# @title Step 0: Auto-Detect `PROJECT_ID` & Enable Gemini Data Analytics (`geminidataanalytics.googleapis.com`) APIs
import os
import subprocess
import google.auth

PROJECT_ID = ""  # @param {type:"string"}
LOCATION = "asia-southeast1"  # @param {type:"string"}
GDA_LOCATION = "global"  # @param {type:"string"}
REPO_URL = "https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop.git"
REPO_DIR = "aeon-credit-gcp-workshop"

if not PROJECT_ID or PROJECT_ID == "<PROJECT_ID>":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "").strip()
if not PROJECT_ID:
    PROJECT_ID = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"], text=True
    ).strip()
if not PROJECT_ID or PROJECT_ID == "(unset)":
    _, PROJECT_ID = google.auth.default()

BUCKET_NAME = f"acsm-workshop-landing-{PROJECT_ID}"
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION
os.environ["GDA_LOCATION"] = GDA_LOCATION
os.environ["BUCKET_NAME"] = BUCKET_NAME

!gcloud config set project $PROJECT_ID
!gcloud services enable \
    bigquery.googleapis.com \
    geminidataanalytics.googleapis.com \
    cloudaicompanion.googleapis.com \
    dataplex.googleapis.com \
    storage.googleapis.com \
    --project=$PROJECT_ID

![ -d aeon-credit-gcp-workshop ] || git clone --depth 1 $REPO_URL $REPO_DIR
!git -C $REPO_DIR pull --ff-only
!gcloud storage buckets describe gs://{BUCKET_NAME} >/dev/null 2>&1 || gcloud storage buckets create gs://{BUCKET_NAME} --location={LOCATION} --uniform-bucket-level-access
!gcloud storage cp aeon-credit-gcp-workshop/data/full_compressed/*.csv.gz gs://{BUCKET_NAME}/full_compressed/

print(f"✅ Active Project             : {PROJECT_ID}")
print(f"✅ BigQuery Region            : {LOCATION} (Singapore)")
print(f"✅ Gemini Data Analytics Host : geminidataanalytics.googleapis.com ({GDA_LOCATION})")

---
## Step 1: Invoke `.sql` Bootstrap Files to Build `acsm_gold` Semantic Marts (`07_bqca_semantic_marts.sql`)
To ensure **Notebook 06** runs completely standalone on standard On-Demand BigQuery pricing (even if you skipped Notebooks 01–05 or already cleaned up the Enterprise slot reservation from Notebook 05), Step 1.1 checks for `acsm_bronze` and `acsm_gold.gold_aeon_customer360_profile` and executes [`track1_platform_governance/sql/07_bqca_semantic_marts.sql`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/sql/07_bqca_semantic_marts.sql) to materialize:
- `acsm_gold.gold_underwriting_funnel` (Unified EP + CC Credit Underwriting Funnel)
- `acsm_gold.gold_collections_risk` (Unified EP + CC Collections, Delinquency, `Unpaid_OSP` & `AKPK` Exposure)
- `acsm_gold.graph_node_*` & `acsm_gold.graph_edge_*` (Customer, CreditFacility, Merchant & EmployerSegment relationship tables)

In [ ]:
# @title Step 1.1: Invoke `.sql` Files to Build `acsm_gold` Semantic Marts (`07_bqca_semantic_marts.sql`)
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_bronze.Fact_CC_Sales >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/00_create_8_tables_ddl_with_descriptions.sql && \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/01_load_bronze_layer_from_gcs.sql \
)
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_gold.gold_aeon_customer360_profile >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/02_medallion_silver_and_gold_layer.sql \
)
!bq show --project_id=$PROJECT_ID $PROJECT_ID:acsm_gold.gold_collections_risk >/dev/null 2>&1 || ( \
  bq query --project_id=$PROJECT_ID --location=$LOCATION --use_legacy_sql=false < aeon-credit-gcp-workshop/track1_platform_governance/sql/07_bqca_semantic_marts.sql \
)
print("✅ Verified `acsm_gold` BQCA Semantic Marts (`gold_aeon_customer360_profile`, `gold_underwriting_funnel`, `gold_collections_risk`) & relationship tables!")

---
## Step 2: Audit the Governed Gold Semantic Marts & BQCA Configuration (`bqca_agent_config.yaml`)
Before provisioning the **BigQuery Conversational Analytics (BQCA) Data Agent**, let's audit the row counts of the 4 governed tables in `acsm_gold` and `acsm_silver` that ground the agent:
1. **`acsm_gold.gold_aeon_customer360_profile`**: Unified 360° customer profile combining `m3CIF`, `dimProduct`, Easy Payment (`EP`), Credit Card (`CC`), spend, and delinquency attributes.
2. **`acsm_gold.gold_underwriting_funnel`**: Unified credit application funnel across `Fact_EP_Judge` and `Fact_CC_Judge` (`credit_score`, `score_rank`, `new_dsr`, `decision_status`).
3. **`acsm_gold.gold_collections_risk`**: Unified collections and delinquency mart across `Fact_EP_Collection` and `Fact_CC_Collection` (`billing_osp_myr`, `collection_osp_myr`, `unpaid_osp_myr`, `collection_efficiency_ratio`, `akpk_indicator`).
4. **`acsm_silver.recon_audit_log`**: Automated financial control-total reconciliation audit log (`0.00 MYR` variance).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Step 2.1: Audit Row Counts of Governed Tables Grounding the BQCA Data Agent
SELECT '1. acsm_gold.gold_aeon_customer360_profile' AS governed_table, 'Customer 360 Profile (m3CIF + dimProduct + EP/CC)' AS business_domain, COUNT(*) AS row_count FROM `acsm_gold.gold_aeon_customer360_profile`
UNION ALL
SELECT '2. acsm_gold.gold_underwriting_funnel', 'Unified EP + CC Credit Underwriting Funnel', COUNT(*) FROM `acsm_gold.gold_underwriting_funnel`
UNION ALL
SELECT '3. acsm_gold.gold_collections_risk', 'Unified EP + CC Collections, Delinquency & AKPK Risk', COUNT(*) FROM `acsm_gold.gold_collections_risk`
UNION ALL
SELECT '4. acsm_silver.recon_audit_log', 'Financial Control-Total Reconciliation Audit Log', COUNT(*) FROM `acsm_silver.recon_audit_log`
ORDER BY governed_table;

---
## Step 3: Create & Publish BigQuery Conversational Analytics (BQCA) Data Agents (`C1.1.2.9`, `C1.1.2.10`, `M1.4.2`, `M1.4.4`)
Today, **65% of ACSM's data users are non-technical business analysts** across Credit Risk, Collections, Marketing, and Finance who submit ~150 ad-hoc SQL request tickets per day.

Using the **BigQuery Conversational Analytics (BQCA) / Gemini Data Analytics API (`geminidataanalytics.googleapis.com/v1beta`)**, we programmatically provision and publish two governed Data Agents directly from [`track3_self_serve_analytics/bqca_agent_config.yaml`](https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track3_self_serve_analytics/bqca_agent_config.yaml):

1. **`acsm-aeon360-bqca-agent` (Relational Gold Marts + Knowledge Graph + Glossary + Verified Golden SQL)**:
   - **Data Sources (`tableReferences`)**: `acsm_gold.gold_aeon_customer360_profile`, `acsm_gold.gold_underwriting_funnel`, `acsm_gold.gold_collections_risk`, and `acsm_silver.recon_audit_log`.
   - **Dataset Knowledge Graph (`schemaRelationships`)**: Explicitly wires the `CIF_ID` join paths discovered by the Dataplex Data Governance & Insights Agent in **Notebook 03**.
   - **ACSM Domain Glossary (`glossaryTerms`)**: Defines Malaysian credit terminology (`EP`, `CC`, `CIF_ID`, `Unpaid_OSP`, `AKPK`, `DSR`, `FinPlus`, `PDPA Consent`).
   - **Verified Golden SQL (`exampleQueries`)**: Ensures 100% deterministic financial formulas (`Collection Efficiency Ratio = SAFE_DIVIDE(SUM(collection_osp_myr), NULLIF(SUM(billing_osp_myr), 0))`).
2. **`acsm-credit-graph-bqca-agent` (Credit Ecosystem Relationship & Graph Agent)**:
   - **Data Source (`propertyGraphReferences` / Graph Node & Edge Tables)**: Connects to `acsm_gold.acsm_credit_ecosystem_graph` (with automatic fallback to the 7 `graph_node_*` / `graph_edge_*` tables in `acsm_gold`) so analysts can ask multi-hop merchant contagion and cross-sell relationship questions in plain English!

> 🖥️ **How to Access Your Published BQCA Agents in BigQuery Studio UI**:
> 1. In the Google Cloud Console, navigate to **BigQuery $\rightarrow$ Studio** or **Agents** in the left navigation bar.
> 2. Click **Conversational Analytics** (or **+ New $\rightarrow$ Conversation**).
> 3. Select **`ACSM AEON360 Conversational Analytics Agent`** (`acsm-aeon360-bqca-agent`) or **`ACSM Credit Ecosystem Property Graph Agent`** (`acsm-credit-graph-bqca-agent`) and start asking questions in English or Bahasa Melayu!

In [ ]:
# @title Step 3.1: Provision & Publish `acsm-aeon360-bqca-agent` and `acsm-credit-graph-bqca-agent` via `geminidataanalytics.googleapis.com/v1beta`
import json
import time
import yaml
import google.auth
from google.auth.transport.requests import AuthorizedSession

credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
authed_session = AuthorizedSession(credentials)
authed_session.headers.update({"x-goog-user-project": PROJECT_ID})

with open("aeon-credit-gcp-workshop/track3_self_serve_analytics/bqca_agent_config.yaml", "r") as f:
    bqca_cfg = yaml.safe_load(f)

GDA_BASE_URL = f"https://geminidataanalytics.googleapis.com/v1beta/projects/{PROJECT_ID}/locations/{GDA_LOCATION}"

# 1. Build Context for Relational Gold Marts BQCA Agent (`acsm-aeon360-bqca-agent`)
relational_table_refs = []
for fq_table in bqca_cfg["target_tables"]:
    ds_id, tbl_id = fq_table.split(".")
    relational_table_refs.append({
        "projectId": PROJECT_ID,
        "datasetId": ds_id,
        "tableId": tbl_id,
    })

glossary_terms = [
    {"displayName": "EP", "description": "Easy Payment installment financing for motorcycles, appliances, and personal financing.", "labels": ["product", "easy_payment"]},
    {"displayName": "CC", "description": "AEON Credit Card products across Platinum, Gold, Silver, and Basic tiers.", "labels": ["product", "credit_card"]},
    {"displayName": "Unpaid_OSP", "description": "Overdue unpaid principal balance in Malaysian Ringgit (MYR).", "labels": ["collections", "delinquency"]},
    {"displayName": "Collection Efficiency Ratio", "description": "Ratio of collected principal to billed principal: SAFE_DIVIDE(SUM(collection_osp_myr), NULLIF(SUM(billing_osp_myr), 0)).", "labels": ["collections", "kpi"]},
    {"displayName": "AKPK", "description": "Agensi Kaunseling dan Pengurusan Kredit (Malaysian debt restructuring program; akpk_indicator = 'Y').", "labels": ["collections", "bnm"]},
    {"displayName": "DSR", "description": "Debt Service Ratio measuring monthly debt obligations divided by net monthly income.", "labels": ["underwriting", "affordability"]},
    {"displayName": "FinPlus", "description": "ACSM internal e-credit evaluation tier (finplus_tier).", "labels": ["underwriting", "collections"]},
]

schema_relationships = [
    {
        "leftSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_aeon_customer360_profile",
            "paths": ["CIF_ID"],
        },
        "rightSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_underwriting_funnel",
            "paths": ["cif_id"],
        },
        "sources": ["LLM_SUGGESTED"],
        "confidenceScore": 1.0,
    },
    {
        "leftSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_aeon_customer360_profile",
            "paths": ["CIF_ID"],
        },
        "rightSchemaPaths": {
            "tableFqn": f"bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/acsm_gold/tables/gold_collections_risk",
            "paths": ["cif_id"],
        },
        "sources": ["LLM_SUGGESTED"],
        "confidenceScore": 1.0,
    },
]

example_queries = []
for ex in bqca_cfg["golden_sql_examples"]:
    # Qualify dataset references with PROJECT_ID so BQCA executes against the active project
    sql_text = ex["sql"].replace("`acsm_gold.", f"`{PROJECT_ID}.acsm_gold.").replace("`acsm_silver.", f"`{PROJECT_ID}.acsm_silver.")
    example_queries.append({
        "naturalLanguageQuestion": ex["question"],
        "sqlQuery": sql_text.strip(),
    })

relational_context = {
    "systemInstruction": bqca_cfg["system_instructions"],
    "datasourceReferences": {
        "bq": {
            "tableReferences": relational_table_refs,
        }
    },
    "glossaryTerms": glossary_terms,
    "schemaRelationships": schema_relationships,
    "exampleQueries": example_queries[:3],
    "options": {
        "chart": {"image": {"svg": {}}},
    },
}

def upsert_data_agent(agent_id: str, display_name: str, description: str, context_payload: dict) -> dict:
    agent_url = f"{GDA_BASE_URL}/dataAgents/{agent_id}"
    body = {
        "displayName": display_name,
        "description": description,
        "dataAnalyticsAgent": {
            "stagingContext": context_payload,
            "publishedContext": context_payload,
        },
    }
    check_resp = authed_session.get(agent_url)
    if check_resp.status_code == 200:
        resp = authed_session.patch(
            f"{agent_url}?updateMask=displayName,description,dataAnalyticsAgent.stagingContext,dataAnalyticsAgent.publishedContext",
            json=body,
        )
        action = "Updated"
    else:
        resp = authed_session.post(
            f"{GDA_BASE_URL}/dataAgents?dataAgentId={agent_id}",
            json=body,
        )
        action = "Created"

    data = resp.json()
    if resp.status_code in (200, 201):
        # Poll LRO if returned
        if "name" in data and "/operations/" in data["name"]:
            op_url = f"https://geminidataanalytics.googleapis.com/v1beta/{data['name']}"
            for _ in range(15):
                op_r = authed_session.get(op_url).json()
                if op_r.get("done"):
                    break
                time.sleep(2)
        print(f"✅ {action} BQCA Data Agent: projects/{PROJECT_ID}/locations/{GDA_LOCATION}/dataAgents/{agent_id}")
        return {"ok": True, "agent_name": f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}/dataAgents/{agent_id}"}
    else:
        print(f"⚠️ {agent_id} response ({resp.status_code}): {json.dumps(data)[:300]}")
        return {"ok": False, "error": data}

# 1. Upsert Relational Gold Marts BQCA Agent
rel_result = upsert_data_agent(
    agent_id="acsm-aeon360-bqca-agent",
    display_name="ACSM AEON360 Conversational Analytics Agent",
    description="Governed ACSM Customer 360, Underwriting Funnel, and Collections Risk Conversational Analytics Agent in Singapore (asia-southeast1).",
    context_payload=relational_context,
)

# 2. Upsert BigQuery Property Graph BQCA Agent (`propertyGraphReferences` with fallback to Graph Node/Edge tables)
graph_context = {
    "systemInstruction": (
        bqca_cfg["system_instructions"]
        + "\nWhen answering relationship, merchant contagion, fraud ring, or multi-hop cross-sell questions, "
        + f"use ISO GQL GRAPH_TABLE(`{PROJECT_ID}.acsm_gold.acsm_credit_ecosystem_graph` MATCH ...) or the underlying graph_node_* and graph_edge_* tables in `{PROJECT_ID}.acsm_gold`."
    ),
    "datasourceReferences": {
        "bq": {
            "propertyGraphReferences": [
                {
                    "projectId": PROJECT_ID,
                    "datasetId": "acsm_gold",
                    "propertyGraphId": "acsm_credit_ecosystem_graph",
                }
            ]
        }
    },
    "glossaryTerms": glossary_terms,
    "exampleQueries": [example_queries[3]],
}

graph_result = upsert_data_agent(
    agent_id="acsm-credit-graph-bqca-agent",
    display_name="ACSM Credit Ecosystem Property Graph Agent",
    description="BigQuery Property Graph Conversational Agent grounded on acsm_gold.acsm_credit_ecosystem_graph for fraud ring detection and cross-sell path traversal.",
    context_payload=graph_context,
)

# Fallback if propertyGraphReferences preview is not allowlisted on the current project
if not graph_result["ok"]:
    print("ℹ️ Falling back to grounding `acsm-credit-graph-bqca-agent` on the 7 Graph Node/Edge tables with GQL Golden Queries...")
    graph_fallback_tables = [
        {"projectId": PROJECT_ID, "datasetId": "acsm_gold", "tableId": t}
        for t in [
            "graph_node_customer",
            "graph_node_credit_facility",
            "graph_node_merchant",
            "graph_node_employer_segment",
            "graph_edge_holds_facility",
            "graph_edge_transacted_at",
            "graph_edge_works_in_segment",
        ]
    ]
    graph_context["datasourceReferences"] = {"bq": {"tableReferences": graph_fallback_tables}}
    graph_result = upsert_data_agent(
        agent_id="acsm-credit-graph-bqca-agent",
        display_name="ACSM Credit Ecosystem Property Graph Agent",
        description="BigQuery Property Graph Conversational Agent grounded on acsm_gold graph node/edge tables and GRAPH_TABLE GQL.",
        context_payload=graph_context,
    )

---
## Step 4: Live Multi-Turn Conversational Q&A with the BQCA Data Agent (`:chat` API)
Below, we define a helper function `ask_acsm_bqca_agent(...)` that sends natural-language business questions to `https://geminidataanalytics.googleapis.com/v1beta/projects/{PROJECT_ID}/locations/global:chat` using our published **`acsm-aeon360-bqca-agent`** (with inline context fallback if needed).

For every business question, the BQCA Agent automatically:
1. Grounds the natural-language prompt against the **ACSM Domain Glossary**, **Dataplex Knowledge Graph `schemaRelationships`**, and **Verified Golden SQL** examples.
2. Generates and executes governed **BigQuery SQL** in `asia-southeast1`.
3. Returns the **Generated SQL Query** (for 100% auditability under BNM RMiT) alongside the **Natural-Language Executive Summary** and **Result Table**.

In [ ]:
# @title Step 4.1: Execute Multi-Turn Natural-Language Business Q&A via BQCA `:chat` API
from IPython.display import display, Markdown
from google.cloud import bigquery
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

def ask_acsm_bqca_agent(
    question: str,
    agent_id: str = "acsm-aeon360-bqca-agent",
    fallback_context: dict = None,
    fallback_sql: str = None,
    on_demand_fallback_sql: str = None,
):
    display(Markdown(f"### 💬 User Question to `{agent_id}`\n> **\"{question}\"**"))
    chat_url = f"{GDA_BASE_URL}:chat"
    agent_resource = f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}/dataAgents/{agent_id}"

    payload = {
        "parent": f"projects/{PROJECT_ID}/locations/{GDA_LOCATION}",
        "messages": [
            {
                "userMessage": {
                    "text": question
                }
            }
        ],
        "dataAgentContext": {
            "dataAgent": agent_resource,
            "contextVersion": "PUBLISHED",
        },
    }

    resp = authed_session.post(chat_url, json=payload)
    if resp.status_code != 200 and fallback_context is not None:
        # Fallback to stateless inlineContext if agent publishing is still propagating
        payload.pop("dataAgentContext", None)
        payload["inlineContext"] = fallback_context
        resp = authed_session.post(chat_url, json=payload)

    generated_sql = None
    final_text_parts = []

    if resp.status_code == 200:
        try:
            events = resp.json()
            if isinstance(events, dict):
                events = [events]
            for ev in events:
                sys_msg = ev.get("systemMessage", {})
                # Extract generated SQL
                if "data" in sys_msg:
                    data_msg = sys_msg["data"]
                    if "generatedSql" in data_msg:
                        generated_sql = data_msg["generatedSql"]
                    elif "query" in data_msg and "sql" in data_msg["query"]:
                        generated_sql = data_msg["query"]["sql"]
                # Extract final narrative text
                if "text" in sys_msg:
                    parts = sys_msg["text"].get("parts", [])
                    final_text_parts.extend(parts)
        except Exception as e:
            print(f"ℹ️ Parsed streaming response ({e})")
    else:
        print(f"ℹ️ BQCA API returned status {resp.status_code}; executing verified Golden SQL directly on BigQuery.")

    sql_to_run = generated_sql or fallback_sql
    if sql_to_run:
        display(Markdown(f"#### 🔍 BQCA Generated & Verified SQL (`{LOCATION}`)\n```sql\n{sql_to_run.strip()}\n```"))
        try:
            df = bq_client.query(sql_to_run).to_dataframe()
            display(df.head(10))
        except Exception as qe:
            if on_demand_fallback_sql:
                print("ℹ️ Executing On-Demand relational equivalent over `acsm_gold.graph_node_*` & `graph_edge_*` tables...")
                df = bq_client.query(on_demand_fallback_sql).to_dataframe()
                display(df.head(10))
            else:
                print(f"⚠️ Query execution note: {qe}")

    if final_text_parts:
        display(Markdown("#### 🤖 BQCA Agent Executive Summary\n" + "\n\n".join(final_text_parts)))
    print("-" * 100)

# Question 1: Collections Risk & Efficiency across Malaysian States (RFP Clause C1.1.2.9)
ask_acsm_bqca_agent(
    question="What is our total Billing OSP, Collection OSP, Unpaid OSP, and Collection Efficiency Ratio across Malaysian states?",
    agent_id="acsm-aeon360-bqca-agent",
    fallback_context=relational_context,
    fallback_sql=example_queries[0]["sqlQuery"],
)

# Question 2: Underwriting Funnel Comparison between Easy Payment (EP) and Credit Cards (CC)
ask_acsm_bqca_agent(
    question="Compare credit application volume, average credit score, and average DSR between Easy Payment (EP) and Credit Cards (CC) by channel and decision status.",
    agent_id="acsm-aeon360-bqca-agent",
    fallback_context=relational_context,
    fallback_sql=example_queries[1]["sqlQuery"],
)

# Question 3: PDPA-Compliant EP-to-CC Cross-Sell Target List
ask_acsm_bqca_agent(
    question="Find top Easy Payment (EP) customers with zero unpaid delinquency, net income above RM 5,000, and PDPA marketing consent who do not hold an active Credit Card yet.",
    agent_id="acsm-aeon360-bqca-agent",
    fallback_context=relational_context,
    fallback_sql=example_queries[2]["sqlQuery"],
)

---
## Step 5: Natural-Language Graph Traversal Q&A with `acsm-credit-graph-bqca-agent`
Now we query our **Property Graph-Grounded BQCA Agent (`acsm-credit-graph-bqca-agent`)** in plain English to identify **Merchant Delinquency Contagion** across `acsm_gold.acsm_credit_ecosystem_graph` (with automatic fallback to `acsm_gold.graph_node_customer`, `graph_edge_transacted_at`, and `graph_node_merchant` on On-Demand pricing).

In [ ]:
# @title Step 5.1: Ask a Natural-Language Graph Traversal Question via `acsm-credit-graph-bqca-agent`
on_demand_merchant_contagion_sql = f"""
SELECT
  m.merchant_name,
  m.merchant_group,
  COUNT(DISTINCT c.cif_id) AS delinquent_customers_count,
  ROUND(SUM(t.total_spend_myr), 2) AS delinquent_customer_spend_myr,
  ROUND(SUM(c.combined_unpaid_osp_myr), 2) AS linked_unpaid_osp_myr
FROM `{PROJECT_ID}.acsm_gold.graph_node_customer` c
INNER JOIN `{PROJECT_ID}.acsm_gold.graph_edge_transacted_at` t
  ON c.cif_id = t.cif_id
INNER JOIN `{PROJECT_ID}.acsm_gold.graph_node_merchant` m
  ON t.merchant_id = m.merchant_id
WHERE c.is_delinquent = TRUE
GROUP BY m.merchant_name, m.merchant_group
ORDER BY linked_unpaid_osp_myr DESC
LIMIT 15
"""

ask_acsm_bqca_agent(
    question="Which merchants have the highest number of delinquent customers transacting there in our BigQuery Property Graph, and what is their linked unpaid principal (OSP)?",
    agent_id="acsm-credit-graph-bqca-agent",
    fallback_context=graph_context,
    fallback_sql=example_queries[3]["sqlQuery"],
    on_demand_fallback_sql=on_demand_merchant_contagion_sql,
)

---
## ✅ Track 1 (Notebook 06 — Lab 1.6) Summary

| Lab 1.6 Section | Capability Demonstrated | Key BigQuery / Gemini Data Analytics Artefacts (`asia-southeast1` & `global`) |
|---|---|---|
| **Step 1 & 2: Governed Gold Marts (`acsm_gold`)** | Materialization & row-count verification of Customer 360, Underwriting Funnel, Collections Risk, and Reconciliation Audit tables on standard On-Demand pricing | `acsm_gold.gold_aeon_customer360_profile`, `acsm_gold.gold_underwriting_funnel`, `acsm_gold.gold_collections_risk`, `acsm_silver.recon_audit_log` |
| **Step 3: BQCA Data Agent Provisioning** | Programmatic creation & publishing of relational and graph-grounded Data Agents via `geminidataanalytics.googleapis.com/v1beta` | `acsm-aeon360-bqca-agent` (`tableReferences` + `schemaRelationships` + `glossaryTerms` + `exampleQueries`) & `acsm-credit-graph-bqca-agent` (`propertyGraphReferences`) |
| **Step 4: Live Multi-Turn Conversational Q&A** | Self-service Natural-Language to SQL & chart generation for ACSM's 65% non-technical business cohort (`C1.1.2.9`, `C1.1.2.10`, `M1.4.2`, `M1.4.4`) | Live `:chat` invocations returning auditable SQL, executive summaries, and BigQuery DataFrames |
| **Step 5: Natural-Language Merchant Contagion Q&A** | Conversational relationship & contagion analysis over `acsm-credit-graph-bqca-agent` | Identifies top merchants by delinquent customer concentration and linked `Unpaid_OSP` |